# Printer calibration (dispenser)

Same workflow as the lab notebook: connect → home → jog → local origin → build `tube_coordinates.json`.

Requires: `pip install -e ../..` from the repo root and `pip install pyserial`.
Optional for the widget editor: `ipywidgets`.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
ROOT = HERE.parents[1] if HERE.name == "printer_calibration" else HERE
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from bayesian_optimization.microfluidics import (
    Printer3DClient,
    TubeCoordinatesGenerator,
    create_interactive_generator,
)

COM_PORT = "COM11"  # Device Manager
COORD_FILE = HERE / "tube_coordinates.json"
SAFE_Z = 35.0

## 1. Connect, home, safe Z

In [ ]:
printer = Printer3DClient(port=COM_PORT, baudrate=115200)
printer.connect()
printer.home()
printer.move_safe(SAFE_Z)

## 2. Relative jog

Move until the tip is at the point you want as local origin (or a known rack corner). Uncomment / edit steps as needed.

In [ ]:
# printer.jog_x(-120)
# printer.jog_x(-1)
# printer.jog_y(110)
# printer.jog_y(-0.5)
printer.jog_z(50)
# printer.jog_z(-0.2)

## 3. Local origin and rack geometry

`set_local_origin_here` issues G92 so the current tip position becomes X0 Y0.

In [ ]:
printer.set_local_origin_here(z_value=0.0)
printer.set_small_rack_geometry(
    pitch_x=30.0,
    pitch_y=30.0,
    cols=10,
    rows=5,
)
# printer.set_big_tube_positions({
#     1: (20.0, 95.0),
#     2: (70.0, 95.0),
#     3: (115.0, 95.0),
# })
printer.move_safe(SAFE_Z, z_feed=1200)

# Optional smoke test in local coordinates:
# printer.goto_local_point(x=10.0, y=10.0, z=0.0, xy_feed=5000, z_feed=1200)
# printer.goto_local_zero(xy_feed=3000, z_feed=1200)

## 4. Tube map for ScheduleRunner

JSON format: `{"1": [x, y], "2": [x, y], ...}` in **local** coordinates.

In [ ]:
gen = create_interactive_generator(COORD_FILE)

In [ ]:
# Example rack (edit to match your plate). Indices are schedule tube ids.
gen.add_tube(1, -90.0, -500.0)  # big tube
gen.add_tube(2, -40.0, -500.0)
gen.add_tube(3, 0.0, -500.0)
gen.add_tube(4, 15.0, -135.0)
gen.add_tube(5, 0.0, -135.0)
gen.add_tube(6, -15.0, -135.0)
gen.add_tube(7, -30.0, -135.0)
gen.add_tube(8, -45.0, -135.0)
gen.add_tube(9, -60.0, -135.0)
gen.add_tube(10, -75.0, -135.0)
gen.add_tube(11, -90.0, -135.0)
gen.add_tube(12, -105.0, -135.0)
gen.add_tube(13, -120.0, -135.0)

gen.list_all()
gen.save()

In [ ]:
gen2 = TubeCoordinatesGenerator(COORD_FILE)
gen2.list_all()
coord = gen2.get_tube(1)
if coord:
    print(f"Tube 1: X={coord[0]}, Y={coord[1]}")

## 5. Use with ScheduleRunner

After pump connect (`MF`) and a loaded schedule DataFrame:

```python
from bayesian_optimization.microfluidics import ScheduleRunner

runner = ScheduleRunner(
    MF,
    pumps=(0, 1, 2, 3, 4, 5, 6),
    printer=printer,
    coordinates_file=str(COORD_FILE),
    tick_sleep=0.01,
)
runner.PRE_PUMP_DELAY = 7
runner.XY_SETTLE_DELAY = 0.1
runner.Z_SETTLE_DELAY = 0.1
runner.load(df_schedule)
await runner.run()
```

Pumps only: pass `printer=None`.

In [ ]:
# printer.disconnect()